<a href="https://colab.research.google.com/github/afullhart/climateanalogs/blob/main/Colab/Accuracy_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
%reset -f

In [9]:
!pip install rioxarray

In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths (Using your established working directories)
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
elc_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Safely map Original Cluster ID to Temperature Order using the Zonal CSV
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters using native rasterio
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the ER3 codes (US_L3CODE) to your 5 Diagnostic Groups
er3_to_macro = {
    14: 1, 81: 1,
    13: 2, 20: 2, 22: 2, 24: 2,
    5: 3, 18: 3, 19: 3, 21: 3, 80: 3,
    23: 4, 79: 4,
    25: 5, 26: 5
}

elc_gdf = gpd.read_file(elc_shapefile_path)
elc_gdf['Macro_ID'] = elc_gdf['US_L3CODE'].astype(int).map(er3_to_macro)
elc_gdf = elc_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the ELC polygons to perfectly align with the ISODATA grid
elc_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(elc_gdf.geometry, elc_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix (In-Memory Contingency Table)
overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Case-Normalized Groups (Temperature Ordered)
cn_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2, 3]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11, 12, 14]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 13, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5, 13]},
    'Great Plains': {'macro_id': 5, 'temp_clusters': [4, 11]}
}

# 9. Calculate Metrics Dynamically to construct Table 3
results = []
for group_name, params in cn_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    # Translate temperature orders back to original raster IDs for matrix indexing
    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    # Extract Base Area Footprints
    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    # Derive the Confusion Matrix
    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    # Compute Final Evaluation Metrics
    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

# Output the final metrics exactly like Table 3
metrics_df = pd.DataFrame(results)

print("Table 3: ERA3 Diagnostic Group Accuracy Scores (Case-Normalized Optimization)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3: ERA3 Diagnostic Group Accuracy Scores (Case-Normalized Optimization)
            Diagnostic Group               Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                    [1, 2, 3] 193286 1320108 102075   7969               0.9322                0.9443
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12, 14] 773312  592777 138818 118531               0.8415                0.8387
     Northern Basin/Mountain             [12, 13, 14, 15] 161982 1133317 316935  11204               0.7979                0.8584
            Mogollon/Madrean                   [3, 5, 13] 183855 1282208 131012  26363               0.9031                0.8909
                Great Plains                      [4, 11] 135118 1407027  69475  11818               0.9499                0.9363


In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
orig_to_temp_map = {original_idx + 1: rank + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the MLRARSYM codes to the 6 Diagnostic Groups
mlra_groups = {
    1: ['30', '40'],                                                               # Hot Desert
    2: ['24', '27', '28A', '29', '34A', '34B', '35', '42B'],                       # Great Basin/Colorado Plateau
    3: ['11', '13', '22A', '23', '25', '26', '28B', '36', '46', '47', '48A', '51'],# Northern Basin/Mountain
    4: ['38', '41'],                                                               # Mogollon/Madrean
    5: ['39'],                                                                     # Northern Transition
    6: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']                    # Great Plains
}

mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}
macro_names = {
    1: "Hot Desert",
    2: "Great Basin/Colorado Plateau",
    3: "Northern Basin/Mountain",
    4: "Mogollon/Madrean",
    5: "Northern Transition",
    6: "Great Plains"
}

# Load MLRA shapefile and map codes
mlra_gdf = gpd.read_file(mlra_shapefile_path)
mlra_gdf['Macro_ID'] = mlra_gdf['MLRARSYM'].map(mlra_to_macro)
mlra_gdf = mlra_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the MLRA polygons to perfectly align with the ISODATA grid
mlra_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(mlra_gdf.geometry, mlra_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (mlra_arr > 0)
iso_flat = iso_arr[valid_mask]
mlra_flat = mlra_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix and Pre-calculate areas
overlap_matrix = pd.crosstab(iso_flat, mlra_flat)
total_valid_pixels = len(iso_flat)
cluster_areas = pd.Series(iso_flat).value_counts()
mlra_areas = pd.Series(mlra_flat).value_counts()

# 8. Optimize & Calculate Metrics Dynamically for MLRAs
results = []
for macro_id, group_name in macro_names.items():
    if macro_id not in overlap_matrix.columns:
        continue

    elc_total_area = mlra_areas[macro_id]
    outside_total_area = total_valid_pixels - elc_total_area

    assigned_for_normalized = []

    # --- Case-Normalized Optimization Loop ---
    for cluster_id in overlap_matrix.index:
        overlap_area = overlap_matrix.loc[cluster_id, macro_id] if macro_id in overlap_matrix.columns else 0
        cluster_total_area = cluster_areas[cluster_id]
        outside_spill = cluster_total_area - overlap_area

        tpr_gain = overlap_area / elc_total_area
        tnr_loss = outside_spill / outside_total_area

        # Assign if the True Positive Rate gain exceeds the True Negative Rate loss
        if tpr_gain > tnr_loss:
            assigned_for_normalized.append(cluster_id)

    # Calculate final metrics for the optimized MLRA group
    temp_clusters = sorted([orig_to_temp_map[c] for c in assigned_for_normalized])

    elc_area = elc_total_area
    outside_area = outside_total_area
    cluster_area = cluster_areas[cluster_areas.index.isin(assigned_for_normalized)].sum() if len(assigned_for_normalized) > 0 else 0

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(assigned_for_normalized), macro_id].sum() if len(assigned_for_normalized) > 0 else 0
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

# Output the final MLRA metrics
metrics_df = pd.DataFrame(results)

print("Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Case-Normalized Optimization)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Case-Normalized Optimization)
            Diagnostic Group           Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                [1, 2, 3] 175062 1320489 120299   7588               0.9212                0.9375
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12] 558116  750999 202087 112236               0.8064                0.8103
     Northern Basin/Mountain     [10, 12, 13, 14, 15] 317140  939908 334054  32336               0.7743                0.8226
            Mogollon/Madrean               [3, 5, 13] 132546 1295971 182321  12600               0.8799                0.8949
         Northern Transition              [5, 12, 13]  66553 1264568 280844  11473               0.8199                0.8356
                Great Plains               [4, 5, 11] 179632 1297759 127891  18156               0.9100                0.9092


In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
elc_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the ER3 codes (US_L3CODE) to your 5 Diagnostic Groups
er3_to_macro = {
    14: 1, 81: 1,
    13: 2, 20: 2, 22: 2, 24: 2,
    5: 3, 18: 3, 19: 3, 21: 3, 80: 3,
    23: 4, 79: 4,
    25: 5, 26: 5
}

elc_gdf = gpd.read_file(elc_shapefile_path)
elc_gdf['Macro_ID'] = elc_gdf['US_L3CODE'].astype(int).map(er3_to_macro)
elc_gdf = elc_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the ERA3 polygons to perfectly align with the ISODATA grid
elc_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(elc_gdf.geometry, elc_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix
overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Manual Groups from Table 3 (Temperature Ordered)
manual_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11, 12]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5, 13]},
    'Great Plains': {'macro_id': 5, 'temp_clusters': [4, 11]}
}

# 9. Calculate Metrics Dynamically to construct Table 3
results = []
for group_name, params in manual_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

metrics_df = pd.DataFrame(results)

print("Table 3: ERA3 Diagnostic Group Accuracy Scores (Manual Assignments)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3: ERA3 Diagnostic Group Accuracy Scores (Manual Assignments)
            Diagnostic Group           Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                   [1, 2] 162390 1420971   1212  38865               0.9753                0.9030
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12] 678312  649704  81891 213531               0.8180                0.8243
     Northern Basin/Mountain             [12, 14, 15] 144016 1195529 254723  29170               0.8251                0.8280
            Mogollon/Madrean               [3, 5, 13] 183855 1282208 131012  26363               0.9031                0.8909
                Great Plains                  [4, 11] 135118 1407027  69475  11818               0.9499                0.9363


In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the MLRARSYM codes to the 6 Diagnostic Groups
mlra_groups = {
    1: ['30', '40'],
    2: ['24', '27', '28A', '29', '34A', '34B', '35', '42B'],
    3: ['11', '13', '22A', '23', '25', '26', '28B', '36', '46', '47', '48A', '51'],
    4: ['38', '41'],
    5: ['39'],
    6: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']
}

mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}

mlra_gdf = gpd.read_file(mlra_shapefile_path)
mlra_gdf['Macro_ID'] = mlra_gdf['MLRARSYM'].map(mlra_to_macro)
mlra_gdf = mlra_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the MLRA polygons to perfectly align with the ISODATA grid
mlra_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(mlra_gdf.geometry, mlra_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (mlra_arr > 0)
iso_flat = iso_arr[valid_mask]
mlra_flat = mlra_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix
overlap_matrix = pd.crosstab(iso_flat, mlra_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Manual Groups from Table 3 (Temperature Ordered)
manual_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5]},
    'Northern Transition': {'macro_id': 5, 'temp_clusters': [13]},
    'Great Plains': {'macro_id': 6, 'temp_clusters': [4, 11]}
}

# 9. Calculate Metrics Dynamically for MLRAs
results = []
for group_name, params in manual_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum() if len(orig_clusters) > 0 else 0

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum() if len(orig_clusters) > 0 else 0
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

metrics_df = pd.DataFrame(results)

print("Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Manual Assignments)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Manual Assignments)
            Diagnostic Group       Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert               [1, 2] 156511 1433697   7091  26139               0.9795                0.9260
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11] 486595  843767 109319 183757               0.8195                0.8056
     Northern Basin/Mountain         [12, 14, 15] 256370 1131593 142369  93106               0.8550                0.8109
            Mogollon/Madrean               [3, 5] 122900 1366503 111789  22246               0.9174                0.8856
         Northern Transition                 [13]  32881 1498115  47297  45145               0.9431                0.6954
                Great Plains              [4, 11] 164420 1385477  40173  33368               0.9547                0.9016


In [11]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
elc_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the ER3 codes (US_L3CODE) to your 5 Diagnostic Groups (Final Optimum)
er3_to_macro = {
    14: 1, 81: 1,                                   # Hot Desert
    13: 2, 20: 2, 22: 2, 24: 2,                     # Great Basin/Colorado Plateau
    5: 3, 18: 3, 19: 3, 21: 3, 80: 3,               # Northern Basin/Mountain
    23: 4, 79: 4,                                   # Mogollon/Madrean
    25: 5, 26: 5                                    # Great Plains
}

elc_gdf = gpd.read_file(elc_shapefile_path)
elc_gdf['Macro_ID'] = elc_gdf['US_L3CODE'].astype(int).map(er3_to_macro)
elc_gdf = elc_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the ERA3 polygons to perfectly align with the ISODATA grid
elc_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(elc_gdf.geometry, elc_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix
overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Final Global Optimum Groups (Balanced Accuracy Optimization)
final_diagnostic_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2, 3]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11, 12, 14]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 13, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5, 13]},
    'Great Plains': {'macro_id': 5, 'temp_clusters': [4, 11]}
}

# 9. Calculate Metrics Dynamically
results = []
for group_name, params in final_diagnostic_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

metrics_df = pd.DataFrame(results)

print("Table 3: ER3 Diagnostic Group Accuracy Scores (Fully Unsupervised Global Optimum)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3: ER3 Diagnostic Group Accuracy Scores (Fully Unsupervised Global Optimum)
            Diagnostic Group               Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                    [1, 2, 3] 193286 1320108 102075   7969               0.9322                0.9443
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12, 14] 773312  592777 138818 118531               0.8415                0.8387
     Northern Basin/Mountain             [12, 13, 14, 15] 161982 1133317 316935  11204               0.7979                0.8584
            Mogollon/Madrean                   [3, 5, 13] 183855 1282208 131012  26363               0.9031                0.8909
                Great Plains                      [4, 11] 135118 1407027  69475  11818               0.9499                0.9363


In [12]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the MLRARSYM codes to the 5 Diagnostic Groups (Final Optimum)
mlra_groups = {
    1: ['30', '40'],                                                               # Hot Desert
    2: ['23', '24', '26', '27', '28A', '28B', '29', '34A', '34B', '35', '36', '42B'],# Great Basin/Colorado Plateau
    3: ['11', '13', '22A', '25', '46', '47', '48A', '51'],                         # Northern Basin/Mountain
    4: ['38', '39', '41'],                                                         # Mogollon/Madrean
    5: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']                    # Great Plains
}

mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}

# Load MLRA shapefile and map codes
mlra_gdf = gpd.read_file(mlra_shapefile_path)
mlra_gdf['Macro_ID'] = mlra_gdf['MLRARSYM'].map(mlra_to_macro)
mlra_gdf = mlra_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the MLRA polygons to perfectly align with the ISODATA grid
mlra_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(mlra_gdf.geometry, mlra_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (mlra_arr > 0)
iso_flat = iso_arr[valid_mask]
mlra_flat = mlra_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix
overlap_matrix = pd.crosstab(iso_flat, mlra_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Final Global Optimum Groups (Balanced Accuracy Optimization)
final_diagnostic_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2, 3]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11, 12, 14]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 13, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5, 13]},
    'Great Plains': {'macro_id': 5, 'temp_clusters': [4, 5, 11]}
}

# 9. Calculate Metrics Dynamically
results = []
for group_name, params in final_diagnostic_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

metrics_df = pd.DataFrame(results)

print("Table 3: MLRA Diagnostic Group Accuracy Scores (Fully Unsupervised Global Optimum)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3: MLRA Diagnostic Group Accuracy Scores (Fully Unsupervised Global Optimum)
            Diagnostic Group               Macro-clusters     TP      TN     FP    FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                    [1, 2, 3] 175062 1320489 120299  7588               0.9212                0.9375
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12, 14] 749190  629827 162940 81481               0.8494                0.8482
     Northern Basin/Mountain             [12, 13, 14, 15] 177895 1133259 301022 11262               0.8076                0.8653
            Mogollon/Madrean                   [3, 5, 13] 186805 1272204 128062 36367               0.8987                0.8728
                Great Plains                   [4, 5, 11] 179632 1297759 127891 18156               0.9100                0.9092


In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
eco3_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Load the raw ERA3 Shapefile and rasterize all unique ELCs natively
elc_gdf = gpd.read_file(eco3_shapefile_path)
elc_gdf['US_L3CODE'] = elc_gdf['US_L3CODE'].astype(int)

elc_arr = rasterize(
    shapes=((geom, value) for geom, value in zip(elc_gdf.geometry, elc_gdf['US_L3CODE'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 5. Mask and flatten to generate the master Spatial Overlap Matrix
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 6. Define Fixed Macro-Clusters (Literal Standard Accuracy Optimums for 5 Groups)
fixed_macro_clusters = {
    'Hot Desert': [1, 2],
    'Great Basin/Colorado Plateau': [6, 7, 8, 9, 10, 11, 12, 14],
    'Northern Basin/Mountain': [15],
    'Mogollon/Madrean': [5, 13],
    'Great Plains': [4]
}

# 7. Reverse Optimization: Floating the ELCs
results = []
for group_name, temp_clusters in fixed_macro_clusters.items():
    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    # Calculate the fixed cluster footprint area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    assigned_er3s = []

    # --- Standard Accuracy Optimization Logic ---
    for er3_id in overlap_matrix.columns:
        er3_total_area = overlap_matrix[er3_id].sum()
        overlap_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), er3_id].sum()
        outside_spill = er3_total_area - overlap_area

        # An ER3 strictly maximizes Standard Accuracy if TP gained > TN lost (i.e., >50% overlap)
        if overlap_area > outside_spill:
            assigned_er3s.append(er3_id)

    # Calculate final metrics for the new dynamic diagnostic group
    assigned_er3s.sort()

    elc_area = overlap_matrix[assigned_er3s].sum().sum() if len(assigned_er3s) > 0 else 0
    outside_area = total_valid_pixels - elc_area

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), assigned_er3s].sum().sum() if len(assigned_er3s) > 0 else 0
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    accuracy = (true_positives + true_negatives) / total_valid_pixels if total_valid_pixels > 0 else 0

    results.append({
        'Diagnostic Group': group_name,
        'Fixed Macro-clusters': str(temp_clusters),
        'Optimized ER3s': str(assigned_er3s),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4)
    })

# 8. Output the final metrics
metrics_df = pd.DataFrame(results)

print("Table: Reverse Optimized ER3 Assignments (Standard Accuracy Objective)")
print("="*115)
print(metrics_df.to_string(index=False))


Table: Reverse Optimized ER3 Assignments (Standard Accuracy Objective)
            Diagnostic Group         Fixed Macro-clusters       Optimized ER3s     TP      TN     FP    FN  Prediction Accuracy
                  Hot Desert                       [1, 2]             [14, 81] 162390 1420971   1212 38865               0.9753
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12, 14] [13, 18, 20, 22, 80] 798853  652187 113277 59121               0.8938
     Northern Basin/Mountain                         [15]              [5, 19]  47101 1520779  35422 20136               0.9658
            Mogollon/Madrean                      [5, 13]                 [23] 116259 1400635  66849 39695               0.9344
                Great Plains                          [4]             [25, 26] 118642 1444828  31674 28294               0.9631


In [ ]:
import pandas as pd
import geopandas as gpd

# 1. Define Paths
eco3_shp_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'

# 2. Define Diagnostic Group Mappings (Fully Unsupervised Global Optima)
er3_groups = {
    1: [14, 81],                                   # Hot Desert
    2: [13, 20, 22, 24],                           # Great Basin/Colorado Plateau
    3: [5, 18, 19, 21, 80],                        # Northern Basin/Mountain
    4: [23, 79],                                   # Mogollon/Madrean
    5: [25, 26]                                    # Great Plains
}
er3_to_macro = {code: macro_id for macro_id, codes in er3_groups.items() for code in codes}

mlra_groups = {
    1: ['30', '40'],                                                               # Hot Desert
    2: ['23', '24', '26', '27', '28A', '28B', '29', '34A', '34B', '35', '36', '42B'],# Great Basin/Colorado Plateau
    3: ['11', '13', '22A', '25', '46', '47', '48A', '51'],                         # Northern Basin/Mountain
    4: ['38', '39', '41'],                                                         # Mogollon/Madrean
    5: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']                    # Great Plains
}
mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}

# Consolidated shared naming dictionary for perfect symmetry
macro_names = {
    1: "Hot Desert",
    2: "Great Basin/Colorado Plateau",
    3: "Northern Basin/Mountain",
    4: "Mogollon/Madrean",
    5: "Great Plains"
}

def calculate_boundary_metrics(shp_path, id_column, mapping_dict, names_dict, is_numeric=False):
    """Calculates total vs unique boundary lengths for a given shapefile and mapping."""

    # Load shapefile
    gdf = gpd.read_file(shp_path)

    # Project to NAD83 / Conus Albers (EPSG:5070) for accurate distance measurements in meters
    gdf = gdf.to_crs("EPSG:5070")

    # Map ELCs to Diagnostic Groups
    if is_numeric:
        gdf['Macro_ID'] = gdf[id_column].astype(int).map(mapping_dict)
    else:
        gdf['Macro_ID'] = gdf[id_column].map(mapping_dict)

    gdf = gdf.dropna(subset=['Macro_ID'])

    results = []

    # Iterate through each valid diagnostic group
    for macro_id in sorted(gdf['Macro_ID'].unique()):
        group_gdf = gdf[gdf['Macro_ID'] == macro_id]

        # 1. Total Boundary Length: Sum of individual ELC perimeters (includes shared internal borders)
        total_length_m = group_gdf.geometry.length.sum()

        # 2. Unique Boundary Length: Perimeter after dissolving internal borders
        dissolved_gdf = group_gdf.dissolve(by='Macro_ID')
        unique_length_m = dissolved_gdf.geometry.length.iloc[0]

        # Calculate Percentage
        unique_pct = (unique_length_m / total_length_m) * 100 if total_length_m > 0 else 0

        results.append({
            'Diagnostic Group': names_dict[macro_id],
            'Total Boundary (km)': round(total_length_m / 1000, 2),
            'Unique Exterior Boundary (km)': round(unique_length_m / 1000, 2),
            'Unique % of Total': round(unique_pct, 2)
        })

    return pd.DataFrame(results)

# 3. Execute for ER3 and MLRA
er3_metrics = calculate_boundary_metrics(eco3_shp_path, 'US_L3CODE', er3_to_macro, macro_names, is_numeric=True)
mlra_metrics = calculate_boundary_metrics(mlra_shapefile_path, 'MLRARSYM', mlra_to_macro, macro_names, is_numeric=False)

# 4. Output Results
print("ER3 Optimized Boundary Precision Baseline (10% Floor)")
print("="*75)
print(er3_metrics.to_string(index=False))
print("\n")
print("MLRA Optimized Boundary Precision Baseline (10% Floor)")
print("="*75)
print(mlra_metrics.to_string(index=False))

ER3 Optimized Boundary Precision Baseline (10% Floor)
            Diagnostic Group  Total Boundary (km)  Unique Exterior Boundary (km)  Unique % of Total
                  Hot Desert              5883.71                        5601.14              95.20
Great Basin/Colorado Plateau             16811.13                       14688.10              87.37
     Northern Basin/Mountain              7811.13                        7164.86              91.73
            Mogollon/Madrean              9237.45                        8717.33              94.37
                Great Plains              4939.52                        2443.08              49.46


MLRA Optimized Boundary Precision Baseline (10% Floor)
            Diagnostic Group  Total Boundary (km)  Unique Exterior Boundary (km)  Unique % of Total
                  Hot Desert              3062.84                        2785.31              90.94
Great Basin/Colorado Plateau             16697.97                        8215.16         

In [ ]:
import pandas as pd
import geopandas as gpd

# 1. Define Paths
eco3_shp_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'

# 2. Define Diagnostic Group Mappings (Fully Unsupervised Global Optima)
er3_groups = {
    1: [14, 81],                                   # Hot Desert
    2: [13, 20, 22, 24],                           # Great Basin/Colorado Plateau
    3: [5, 18, 19, 21, 80],                        # Northern Basin/Mountain
    4: [23, 79],                                   # Mogollon/Madrean
    5: [25, 26]                                    # Great Plains
}
er3_to_macro = {code: macro_id for macro_id, codes in er3_groups.items() for code in codes}

mlra_groups = {
    1: ['30', '40'],                                                               # Hot Desert
    2: ['23', '24', '26', '27', '28A', '28B', '29', '34A', '34B', '35', '36', '42B'],# Great Basin/Colorado Plateau
    3: ['11', '13', '22A', '25', '46', '47', '48A', '51'],                         # Northern Basin/Mountain
    4: ['38', '39', '41'],                                                         # Mogollon/Madrean
    5: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']                    # Great Plains
}
mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}

# Consolidated shared naming dictionary for perfect symmetry
macro_names = {
    1: "Hot Desert",
    2: "Great Basin/Colorado Plateau",
    3: "Northern Basin/Mountain",
    4: "Mogollon/Madrean",
    5: "Great Plains"
}

def calculate_boundary_metrics(shp_path, id_column, mapping_dict, names_dict, is_numeric=False):
    """Calculates total vs unique boundary lengths for a given shapefile and mapping."""

    # Load shapefile
    gdf = gpd.read_file(shp_path)

    # Project to NAD83 / Conus Albers (EPSG:5070) for accurate distance measurements in meters
    gdf = gdf.to_crs("EPSG:5070")

    # Map ELCs to Diagnostic Groups
    if is_numeric:
        gdf['Macro_ID'] = gdf[id_column].astype(int).map(mapping_dict)
    else:
        gdf['Macro_ID'] = gdf[id_column].map(mapping_dict)

    gdf = gdf.dropna(subset=['Macro_ID'])

    results = []

    # Iterate through each valid diagnostic group
    for macro_id in sorted(gdf['Macro_ID'].unique()):
        group_gdf = gdf[gdf['Macro_ID'] == macro_id]

        # 1. Total Boundary Length: Sum of individual ELC perimeters (includes shared internal borders)
        total_length_m = group_gdf.geometry.length.sum()

        # 2. Unique Boundary Length: Perimeter after dissolving internal borders
        dissolved_gdf = group_gdf.dissolve(by='Macro_ID')
        unique_length_m = dissolved_gdf.geometry.length.iloc[0]

        # Calculate Percentage
        unique_pct = (unique_length_m / total_length_m) * 100 if total_length_m > 0 else 0

        results.append({
            'Diagnostic Group': names_dict[macro_id],
            'Total Boundary (km)': round(total_length_m / 1000, 2),
            'Unique Exterior Boundary (km)': round(unique_length_m / 1000, 2),
            'Unique % of Total': round(unique_pct, 2)
        })

    return pd.DataFrame(results)

# 3. Execute for ER3 and MLRA
er3_metrics = calculate_boundary_metrics(eco3_shp_path, 'US_L3CODE', er3_to_macro, macro_names, is_numeric=True)
mlra_metrics = calculate_boundary_metrics(mlra_shapefile_path, 'MLRARSYM', mlra_to_macro, macro_names, is_numeric=False)

# 4. Output Results
print("ER3 Optimized Boundary Precision Baseline (10% Floor)")
print("="*75)
print(er3_metrics.to_string(index=False))
print("\n")
print("MLRA Optimized Boundary Precision Baseline (10% Floor)")
print("="*75)
print(mlra_metrics.to_string(index=False))

ER3 Optimized Boundary Precision Baseline (10% Floor)
            Diagnostic Group  Total Boundary (km)  Unique Exterior Boundary (km)  Unique % of Total
                  Hot Desert              5883.71                        5601.14              95.20
Great Basin/Colorado Plateau             14312.09                       12413.08              86.73
     Northern Basin/Mountain              7188.54                        7188.54             100.00
            Mogollon/Madrean              9237.45                        8717.33              94.37
                Great Plains              8061.14                        4047.94              50.22


MLRA Optimized Boundary Precision Baseline (10% Floor)
            Diagnostic Group  Total Boundary (km)  Unique Exterior Boundary (km)  Unique % of Total
                  Hot Desert              3062.84                        2785.31              90.94
Great Basin/Colorado Plateau             15451.53                        8805.57         

In [ ]:
import pandas as pd
import geopandas as gpd

# 1. Define Path
eco3_shp_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'

# 2. Define Diagnostic Group Mappings
er3_to_macro = {
    14: 1, 81: 1,
    13: 2, 20: 2, 22: 2, 24: 2,
    5: 3, 18: 3, 19: 3, 21: 3, 80: 3,
    23: 4, 79: 4,
    25: 5, 26: 5
}

macro_names_er3 = {
    1: "Hot Desert",
    2: "Great Basin/Colorado Plateau",
    3: "Northern Basin/Mountain",
    4: "Mogollon/Madrean",
    5: "Great Plains"
}

# 3. Load Shapefile
gdf = gpd.read_file(eco3_shp_path)

# Map ELCs to Diagnostic Groups
gdf['Macro_ID'] = gdf['US_L3CODE'].astype(int).map(er3_to_macro)
gdf['Diagnostic_Group'] = gdf['Macro_ID'].map(macro_names_er3)

# Drop any regions outside your target groups
gdf = gdf.dropna(subset=['Macro_ID'])

# 4. Analyze ER2 vs ER3 Groupings
results = []
for macro_id in sorted(gdf['Macro_ID'].unique()):
    group_name = macro_names_er3[macro_id]
    group_gdf = gdf[gdf['Macro_ID'] == macro_id]

    # Extract unique Level 3 codes and their corresponding Level 2 classifications
    er_mapping = group_gdf[['US_L3CODE', 'US_L3NAME', 'NA_L2CODE', 'NA_L2NAME']].drop_duplicates()

    for _, row in er_mapping.iterrows():
        results.append({
            'Diagnostic Group': group_name,
            'ER3 Code': row['US_L3CODE'],
            'ER3 Name': row['US_L3NAME'],
            'ER2 Code': row['NA_L2CODE'],
            'ER2 Name': row['NA_L2NAME']
        })

# 5. Output Comparison Table
df_results = pd.DataFrame(results)
print("ER3 Diagnostic Groups vs. Official ER2 Classifications")
print("=" * 105)
print(df_results.to_string(index=False))

ER3 Diagnostic Groups vs. Official ER2 Classifications
            Diagnostic Group ER3 Code                     ER3 Name ER2 Code                         ER2 Name
                  Hot Desert       14       Mojave Basin and Range     10.2                     WARM DESERTS
                  Hot Desert       81      Sonoran Basin and Range     10.2                     WARM DESERTS
Great Basin/Colorado Plateau       13      Central Basin and Range     10.1                     COLD DESERTS
Great Basin/Colorado Plateau       20            Colorado Plateaus     10.1                     COLD DESERTS
Great Basin/Colorado Plateau       22   Arizona/New Mexico Plateau     10.1                     COLD DESERTS
Great Basin/Colorado Plateau       24           Chihuahuan Deserts     10.2                     WARM DESERTS
     Northern Basin/Mountain       18                Wyoming Basin     10.1                     COLD DESERTS
     Northern Basin/Mountain       19  Wasatch and Uinta Mountains      6